In [ ]:
import os
import sys
import torch
import librosa
import numpy as np
from pathlib import Path

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
    print("\n✅ GPU READY!")
    device = torch.device('cuda')
    if 'A100' in torch.cuda.get_device_name(0):
        print("🚀 A100 DETECTED!")
else:
    device = torch.device('cpu')

## 📦 Step 2: Install Required Libraries

In [ ]:
# Install audio processing and transformer libraries
!pip install -q transformers librosa soundfile scipy scikit-learn gradio

## 💾 Step 3: Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

os.chdir('/content')

# Clone if not already present
if not Path('multimodal-emotion-recognition').exists():
    !git clone -q https://github.com/Nishvaraj/multimodal-emotion-recognition.git

os.chdir('/content/multimodal-emotion-recognition')
sys.path.insert(0, '/content/multimodal-emotion-recognition/backend/services')

print("✅ Repository ready!")
print("Current directory:", os.getcwd())

## 📊 Step 4: Verify RAVDESS Dataset

In [ ]:
ravdess_path = Path('data/raw/ravdess')

if ravdess_path.exists():
    audio_files = list(ravdess_path.glob('*.wav'))
    print(f"✅ RAVDESS Dataset Found:")
    print(f"   Audio files: {len(audio_files)}")
    print(f"   Total duration: ~{len(audio_files) * 3} seconds (estimated)")

    # Show file format
    print(f"\n   Sample filenames (RAVDESS format):")
    for i, f in enumerate(audio_files[:3]):
        print(f"      {f.name}")
    print(f"      ...")
else:
    print(f"❌ RAVDESS not found at {ravdess_path}")
    print("   Please ensure audio files are in data/raw/ravdess/")

## 🎵 Step 5: Audio Exploration

In [ ]:
import matplotlib.pyplot as plt

# Load and visualize sample audio
audio_files = list(ravdess_path.glob('*.wav'))[:3]

fig, axes = plt.subplots(3, 2, figsize=(14, 10))

for idx, audio_file in enumerate(audio_files):
    # Load audio
    y, sr = librosa.load(audio_file, sr=22050)

    # Waveform
    axes[idx, 0].plot(y[:sr*2])  # First 2 seconds
    axes[idx, 0].set_title(f'Waveform: {audio_file.name}', fontsize=10)
    axes[idx, 0].set_xlabel('Samples')

    # Mel spectrogram
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    im = axes[idx, 1].imshow(mel_spec_db, aspect='auto', origin='lower')
    axes[idx, 1].set_title(f'Mel Spectrogram: {audio_file.name}', fontsize=10)
    axes[idx, 1].set_ylabel('Mel Frequency')
    plt.colorbar(im, ax=axes[idx, 1])

plt.tight_layout()
plt.show()

print("✅ Audio exploration complete!")

## 📝 Step 6: RAVDESS Format Understanding

RAVDESS filename format: `03-02-01-01-01-01-04.wav`

- Filename identifiers:
  - 1st identifier: Modality (01=speech, 02=song)
  - 2nd identifier: Vocal channel (01=male, 02=female)
  - 3rd identifier: Emotion (01=neutral, 02=calm, 03=happy, 04=sad, 05=angry, 06=fearful, 07=disgust, 08=surprised)
  - 4th identifier: Emotional intensity (01=normal, 02=strong)
  - 5th-6th identifiers: Statement and repetition

**For Phase 3**: Extract emotion (3rd identifier) = 8 emotions → Convert to 7 for consistency

In [ ]:
# Parse RAVDESS emotions
ravdess_emotions_map = {
    '01': 'neutral',
    '02': 'calm',
    '03': 'happy',
    '04': 'sad',
    '05': 'angry',
    '06': 'fear',
    '07': 'disgust',
    '08': 'surprise'
}

# Count emotions in dataset
from collections import Counter

emotion_counts = Counter()
for audio_file in ravdess_path.glob('*.wav'):
    emotion_code = audio_file.name.split('-')[2]
    emotion = ravdess_emotions_map.get(emotion_code, 'unknown')
    emotion_counts[emotion] += 1

print("📊 RAVDESS Emotion Distribution:")
for emotion, count in sorted(emotion_counts.items()):
    print(f"   {emotion}: {count} files")

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
emotions_list = list(emotion_counts.keys())
counts_list = list(emotion_counts.values())

ax.bar(emotions_list, counts_list, color='skyblue', edgecolor='navy')
ax.set_ylabel('Count')
ax.set_title('RAVDESS Dataset - Emotion Distribution')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## ✅ Phase 3 Setup Complete!

✅ GPU verified and optimized
✅ RAVDESS dataset loaded and explored
✅ Audio visualization working
✅ Emotion mapping understood

**Next**: Run `PHASE3_COLAB_02_HuBERT_Training.ipynb` to train speech emotion model